# Przetwarzanie języka naturalnego – Laboratorium nr 1
**Podstawowe techniki przetwarzania tekstu, praca z reprezentacją wektorową**

Rozwiązania ćwiczeń samodzielnych 1–5.

Wymagania: Python ≥ 3.12, pakiety `torch`, `ollama`, `numpy`, `tiktoken`; działające narzędzie Ollama z modelem `nomic-embed-text` (`ollama pull nomic-embed-text`) oraz plik `belmont-sherlock-holmes-w-warszawie.txt` w katalogu notebooka.

In [1]:
import re
import os
import urllib.request

NAZWA_PLIKU = "belmont-sherlock-holmes-w-warszawie.txt"
URL = "https://wolnelektury.pl/media/book/txt/belmont-sherlock-holmes-w-warszawie.txt"

# Jeśli pliku nie ma lokalnie, próbujemy pobrać go z Wolnych Lektur
if not os.path.exists(NAZWA_PLIKU):
    urllib.request.urlretrieve(URL, NAZWA_PLIKU)

with open(NAZWA_PLIKU, "r", encoding="utf-8") as plik:
    tekst = plik.read()

print(len(tekst))  # oczekiwany wynik wg instrukcji: 34074

34074


## Ćwiczenie 1 – klasa `ProstyTokenizer`

Najpierw budujemy słownik tokenów (listing 1.2 i 1.3). Do tokenizacji używamy wyrażenia regularnego z listingu 1.2 (zawiera wielokropek `…`), żeby tokenizacja w klasie była spójna z tą, na podstawie której powstał słownik.

In [2]:
WZORZEC = r'([,:;?_!"…().\']|--|\s)'

tokeny = re.split(WZORZEC, tekst)
tokeny = [t.strip() for t in tokeny if t.strip()]
print(len(tokeny))  # oczekiwany wynik: 6693

słowa = sorted(set(tokeny))
słownik = {token: id for id, token in enumerate(słowa)}
print(list(słownik.items())[:10])

6693
[('!', 0), ('(', 1), (')', 2), ('*', 3), (',', 4), ('-', 5), ('--', 6), ('.', 7), ('//artlibre', 8), ('//wolnelektury', 9)]


In [3]:
class ProstyTokenizer:
    def __init__(self, słownik):
        # token -> id
        self.token_do_id = słownik
        # id -> token (odwzorowanie odwrotne)
        self.id_do_token = {id: token for token, id in słownik.items()}

    def zakoduj(self, tekst):
        tokeny = re.split(WZORZEC, tekst)
        tokeny = [t.strip() for t in tokeny if t.strip()]
        return [self.token_do_id[t] for t in tokeny]

    def dekoduj(self, ids):
        tokeny = " ".join([self.id_do_token[i] for i in ids])
        # usunięcie spacji przed znakami interpunkcyjnymi
        return re.sub(r'\s+([,.?_!"()…\']|--|\s)', r'\1', tokeny)

In [4]:
tokenizer = ProstyTokenizer(słownik)
fragment = "Widzę, że irytował się podczas rozmowy. " \
           "Serweta jego zmięta i rzucona na podłogę."
ids = tokenizer.zakoduj(fragment)
print(ids)  # oczekiwany początek: [372, 4, 2315, 793, 1694, 1393, 1621, 7, 302, ...]
print(tokenizer.dekoduj(ids))

[372, 4, 2315, 793, 1694, 1393, 1621, 7, 302, 835, 2231, 783, 1662, 1089, 1411, 7]
Widzę, że irytował się podczas rozmowy. Serweta jego zmięta i rzucona na podłogę.


## Ćwiczenie 2 – token specjalny `<|nierozpoznany|>`

Rozszerzamy słownik o tokeny specjalne (listing 1.5), a następnie modyfikujemy `zakoduj` tak, aby słowa spoza słownika były zastępowane tokenem `<|nierozpoznany|>`.

In [5]:
słowa = list(słowa)
słowa.extend(["<|koniec|>", "<|nierozpoznany|>"])
print(słowa[-5:])
słownik = {token: id for id, token in enumerate(słowa)}

['„zdrowia', '„Śmiała', '…', '<|koniec|>', '<|nierozpoznany|>']


In [6]:
class ProstyTokenizer:
    def __init__(self, słownik):
        self.token_do_id = słownik
        self.id_do_token = {id: token for token, id in słownik.items()}

    def zakoduj(self, tekst):
        tokeny = re.split(WZORZEC, tekst)
        tokeny = [t.strip() for t in tokeny if t.strip()]
        # nieznane tokeny zastępujemy tokenem specjalnym
        tokeny = [t if t in self.token_do_id else "<|nierozpoznany|>"
                  for t in tokeny]
        return [self.token_do_id[t] for t in tokeny]

    def dekoduj(self, ids):
        tokeny = " ".join([self.id_do_token[i] for i in ids])
        return re.sub(r'\s+([,.?_!"()…\']|--|\s)', r'\1', tokeny)

In [7]:
tokenizer = ProstyTokenizer(słownik)
fragment = "Mój przyjaciel otworzył drzwi i potknął się o próg"
ids = tokenizer.zakoduj(fragment)
print(ids)  # oczekiwany wynik: [189, 1529, 1294, 675, 783, 2400, 1694, 1185, 2400]
print(tokenizer.dekoduj(ids))
# oczekiwane: Mój przyjaciel otworzył drzwi i <|nierozpoznany|> się o <|nierozpoznany|>

[189, 1529, 1294, 675, 783, 2398, 1694, 1185, 2398]
Mój przyjaciel otworzył drzwi i <|nierozpoznany|> się o <|nierozpoznany|>


## Ćwiczenie 3 – uproszczony algorytm BPE

Uzupełnienie szkieletu z listingu 1.12. Dodatkowo, na końcu, pokazana jest pętla wykonująca kilka kolejnych scaleń.

In [8]:
surowy_tekst = "set new new renew reset renew"

# częstość występowania słów
czest_slow = {}
for s in surowy_tekst.split():
    czest_slow[s] = czest_slow.get(s, 0) + 1
print(czest_slow)
# wynik: {'set': 1, 'new': 2, 'renew': 2, 'reset': 1}

# początkowy słownik BPE: unikatowe znaki tekstu (razem ze spacją)
bpe_slownik = set(surowy_tekst)

# słowa zapisane jako litery rozdzielone spacjami + token końca słowa
bpe_czest_slow = {}
for s, i in czest_slow.items():
    bpe_czest_slow[" ".join(s) + " <|koniec|>"] = i

print(bpe_czest_slow)
# wynik:
# {'s e t <|koniec|>': 1, 'n e w <|koniec|>': 2,
#  'r e n e w <|koniec|>': 2, 'r e s e t <|koniec|>': 1}

{'set': 1, 'new': 2, 'renew': 2, 'reset': 1}
{'s e t <|koniec|>': 1, 'n e w <|koniec|>': 2, 'r e n e w <|koniec|>': 2, 'r e s e t <|koniec|>': 1}


In [9]:
def częstość_wystapienia(bpe_czest: dict[str, int]):
    pary: dict[tuple[str, str], int] = {}
    for słowo, czestosc in bpe_czest.items():
        symbole = słowo.split()
        # pary sąsiadujących tokenów w granicach jednego słowa
        for a, b in zip(symbole, symbole[1:]):
            pary[(a, b)] = pary.get((a, b), 0) + czestosc
    return pary


czest_par = częstość_wystapienia(bpe_czest_slow)
print(czest_par)

top = sorted(czest_par.items(), key=lambda x: (-x[1], x[0]))
print(top[0][0])  # wynik: ('e', 'w')

{('s', 'e'): 2, ('e', 't'): 2, ('t', '<|koniec|>'): 2, ('n', 'e'): 4, ('e', 'w'): 4, ('w', '<|koniec|>'): 4, ('r', 'e'): 3, ('e', 'n'): 2, ('e', 's'): 1}
('e', 'w')


In [10]:
# Nowy token powstaje z połączenia najczęstszej pary
najczestsza_para = top[0][0]
nowy_token = "".join(najczestsza_para)
bpe_slownik.add(nowy_token)
print(bpe_slownik)  # wynik: {'w', 'ew', 'r', 'n', ' ', 'e', 's', 't'} (kolejność dowolna)

{'r', 's', ' ', 'n', 't', 'e', 'ew', 'w'}


In [11]:
def scal_pary(para_do_scalenia, bpe_czest: dict[str, int]):
    nowe_bpe_czest = {}
    a, b = para_do_scalenia
    for słowo, czestosc in bpe_czest.items():
        symbole = słowo.split()
        nowe_symbole = []
        i = 0
        while i < len(symbole):
            if i < len(symbole) - 1 and symbole[i] == a and symbole[i + 1] == b:
                nowe_symbole.append(a + b)
                i += 2
            else:
                nowe_symbole.append(symbole[i])
                i += 1
        nowe_bpe_czest[" ".join(nowe_symbole)] = czestosc
    return nowe_bpe_czest


nowe_bpe_czest = scal_pary(top[0][0], bpe_czest_slow)
print(nowe_bpe_czest)
# wynik:
# {'s e t <|koniec|>': 1, 'n ew <|koniec|>': 2,
#  'r e n ew <|koniec|>': 2, 'r e s e t <|koniec|>': 1}

{'s e t <|koniec|>': 1, 'n ew <|koniec|>': 2, 'r e n ew <|koniec|>': 2, 'r e s e t <|koniec|>': 1}


In [12]:
# Dodatkowo: kilka kolejnych iteracji BPE
bpe_czest = dict(bpe_czest_slow)
slownik_bpe = set(surowy_tekst)
for krok in range(1, 6):
    pary = częstość_wystapienia(bpe_czest)
    if not pary:
        break
    najlepsza = sorted(pary.items(), key=lambda x: (-x[1], x[0]))[0][0]
    bpe_czest = scal_pary(najlepsza, bpe_czest)
    slownik_bpe.add("".join(najlepsza))
    print(f"Krok {krok}: scalono {najlepsza} -> {''.join(najlepsza)!r}")
    print("   ", bpe_czest)
print(sorted(slownik_bpe))

Krok 1: scalono ('e', 'w') -> 'ew'
    {'s e t <|koniec|>': 1, 'n ew <|koniec|>': 2, 'r e n ew <|koniec|>': 2, 'r e s e t <|koniec|>': 1}
Krok 2: scalono ('ew', '<|koniec|>') -> 'ew<|koniec|>'
    {'s e t <|koniec|>': 1, 'n ew<|koniec|>': 2, 'r e n ew<|koniec|>': 2, 'r e s e t <|koniec|>': 1}
Krok 3: scalono ('n', 'ew<|koniec|>') -> 'new<|koniec|>'
    {'s e t <|koniec|>': 1, 'new<|koniec|>': 2, 'r e new<|koniec|>': 2, 'r e s e t <|koniec|>': 1}
Krok 4: scalono ('r', 'e') -> 're'
    {'s e t <|koniec|>': 1, 'new<|koniec|>': 2, 're new<|koniec|>': 2, 're s e t <|koniec|>': 1}
Krok 5: scalono ('e', 't') -> 'et'
    {'s et <|koniec|>': 1, 'new<|koniec|>': 2, 're new<|koniec|>': 2, 're s et <|koniec|>': 1}
[' ', 'e', 'et', 'ew', 'ew<|koniec|>', 'n', 'new<|koniec|>', 'r', 're', 's', 't', 'w']


## Ćwiczenie 4 – podobieństwo cosinusowe wektorów osadzeń

Wymaga uruchomionego Ollama i pobranego modelu: `ollama pull nomic-embed-text`.

In [13]:
import torch
import numpy as np
import ollama

nazwa_modelu = "nomic-embed-text"

słowa_emb = ["king", "queen", "man", "woman", "car"]
wektory_embedding = {}

for s in słowa_emb:
    rezultat = ollama.embed(model=nazwa_modelu, input=s)
    wektory_embedding[s] = torch.tensor([rezultat["embeddings"][0]])

wektor_krol = wektory_embedding["king"]
wektor_krolowa = wektory_embedding["queen"]
wektor_mezczyzna = wektory_embedding["man"]
wektor_kobieta = wektory_embedding["woman"]
wektor_samochod = wektory_embedding["car"]
print(wektor_krol.shape)

ModuleNotFoundError: No module named 'torch'

In [ ]:
%pip install torch numpy ollama tiktoken

In [ ]:
def podobienstwo(a, b):
    return torch.cosine_similarity(a, b, dim=1).item()

wyniki = {
    "król – królowa":                              podobienstwo(wektor_krol, wektor_krolowa),
    "mężczyzna – kobieta":                         podobienstwo(wektor_mezczyzna, wektor_kobieta),
    "mężczyzna – samochód":                        podobienstwo(wektor_mezczyzna, wektor_samochod),
    "kobieta – samochód":                          podobienstwo(wektor_kobieta, wektor_samochod),
    "(król − królowa) – (mężczyzna − kobieta)":    podobienstwo(wektor_krol - wektor_krolowa,
                                                                 wektor_mezczyzna - wektor_kobieta),
    "(król − mężczyzna + kobieta) – królowa":      podobienstwo(wektor_krol - wektor_mezczyzna + wektor_kobieta,
                                                                 wektor_krolowa),
}

for para, wartosc in wyniki.items():
    print(f"{para:45s} {wartosc:.4f}")

### Wnioski (do uzupełnienia po uruchomieniu)

Wartości zależą od modelu, dlatego poniżej opisano, **jak interpretować** wyniki; po uruchomieniu komórki wyżej warto je zestawić z własnymi liczbami.

* **król – królowa** oraz **mężczyzna – kobieta**: pary pokrewne znaczeniowo (ta sama kategoria pojęć, różnią się głównie płcią) – oczekujemy wysokiego podobieństwa.
* **mężczyzna – samochód** i **kobieta – samochód**: pojęcia z różnych dziedzin – oczekujemy wyraźnie niższych wartości niż dla par pokrewnych. Zbliżone wartości dla obu par sugerują, że model nie wiąże „samochodu” z żadną z płci.
* **(król − królowa) i (mężczyzna − kobieta)**: sprawdzamy, czy różnica „rodzaj męski – żeński” jest w przestrzeni osadzeń podobnym kierunkiem dla obu par. Wysokie podobieństwo oznaczałoby, że relacja jest kodowana liniowo.
* **(król − mężczyzna + kobieta) i królowa**: klasyczna analogia z word2vec. Wysoki wynik oznacza, że arytmetyka wektorów odtwarza relację. Warto pamiętać, że `nomic-embed-text` to model osadzający całe teksty (nie klasyczny word2vec), więc analogie mogą być słabsze niż w modelach słowo-poziomowych.
* **Ogólnie**: podobieństwo cosinusowe mierzy kąt, nie odległość – ignoruje długość wektorów. Osadzenia z wytrenowanego modelu kodują podobieństwo semantyczne, a wartości dla pojedynczych słów bywają ogólnie dość wysokie (osadzenia zajmują wąski stożek przestrzeni), więc istotne są **różnice** między parami, a nie wartości bezwzględne.

## Ćwiczenie 5 – tokenizacja z użyciem `tiktoken`

Instalacja: `pip install tiktoken`.

In [ ]:
# !pip install tiktoken
import tiktoken

tekst_testowy = ("Dwudziesty pierwszy października roku dwa tysiące "
                 "trzydziestego piątek. Miasto Lublin, Czwarta Rzeczpospolita")

enkodery = ["o200k_base", "cl100k_base", "gpt2"]

for nazwa in enkodery:
    enc = tiktoken.get_encoding(nazwa)
    ids = enc.encode(tekst_testowy)
    print(f"=== {nazwa} === liczba tokenów: {len(ids)}")
    print("ids:", ids)
    # fragmenty tekstu odpowiadające poszczególnym tokenom
    fragmenty = [enc.decode_single_token_bytes(i).decode("utf-8", errors="replace")
                 for i in ids]
    print("fragmenty:", fragmenty)
    # sprawdzenie odwracalności
    print("decode == oryginał:", enc.decode(ids) == tekst_testowy)
    print()

**Obserwacje:** nowsze enkodery (`o200k_base`, `cl100k_base`) mają większe słowniki i zwykle dzielą polskie słowa na dłuższe fragmenty niż `gpt2`, więc dają mniej tokenów. Polskie znaki diakrytyczne (np. „ś”, „ą”) w `gpt2` często rozpadają się na pojedyncze bajty (w kolumnie „fragmenty” widać wtedy znak zastępczy `�`), a liczba tokenów jest największa. Wyrazy rzadkie lub fleksyjne (np. „trzydziestego”, „Rzeczpospolita”) są dzielone na podwyrazy.